In [78]:
from typing import List, TypedDict
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

True

In [79]:
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint

In [80]:
from langchain_huggingface import HuggingFaceEmbeddings
from pydantic import BaseModel, Field
from langchain_core.output_parsers import StrOutputParser
from typing import Annotated
from langchain_core.output_parsers import JsonOutputParser


In [81]:
docs = PyPDFLoader("./documents/book1.pdf").load()

In [82]:
split_docs = RecursiveCharacterTextSplitter(chunk_size = 900,chunk_overlap= 100).split_documents(docs)

for d in split_docs:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

In [83]:

print(len(split_docs))

2433


In [84]:
embed_model = HuggingFaceEmbeddings(model= "sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1031.89it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [85]:
vector_store = FAISS.from_documents(split_docs,embed_model)

In [86]:
retriever = vector_store.as_retriever(search_type = "similarity", search_kwargs = {'k': 4})


In [87]:
class State(TypedDict):
    question : str
    docs : list[Document]
    

    strips : list[str]
    kept_strips: list[str]
    refined_context: str
    keep_refined_chunks:list[dict]

    answer : str

In [101]:
llm = ChatAnthropic(
    model="claude-3-5-haiku-20241022",  # হালকা ও দ্রুত মডেল
    anthropic_api_url=os.getenv("ANTHROPIC_BASE_URL"),
    api_key=os.getenv("sk-OnKHLMLluMyWO67CMuHi7YpZgafLEpDu4jGgXgkPodBKjxRn"),
    temperature=0
)

NameError: name 'ChatAnthropic' is not defined

In [89]:
def retrieve(state):
    q = state['question']
    return {'docs': retriever.invoke(q)}

In [90]:
s = {'question':"eita kiser book",'docs':[]}
r = retrieve(s)
n = r['docs']
print(type(n))
print(len(n))
print(n[0])

<class 'list'>
4
page_content='PREFACE ix
Many people have helped by proofreading draft material and providing com-
ments and suggestions, including Shivani Agarwal, C´edric Archambeau, Arik Azran,
Andrew Blake, Hakan Cevikalp, Michael Fourman, Brendan Frey, Zoubin Ghahra-
mani, Thore Graepel, Katherine Heller, Ralf Herbrich, Geoffrey Hinton, Adam Jo-
hansen, Matthew Johnson, Michael Jordan, Eva Kalyvianaki, Anitha Kannan, Julia
Lasserre, David Liu, Tom Minka, Ian Nabney, Tonatiuh Pena, Y uan Qi, Sam Roweis,
Balaji Sanjiya, Toby Sharp, Ana Costa e Silva, David Spiegelhalter, Jay Stokes, Tara
Symeonides, Martin Szummer, Marshall Tappen, Ilkay Ulusoy, Chris Williams, John
Winn, and Andrew Zisserman.
Finally, I would like to thank my wife Jenna who has been hugely supportive
throughout the several years it has taken to write this book.
Chris Bishop
Cambridge
February 2006' metadata={'producer': 'Acrobat Distiller 6.0 (Windows)', 'creator': 'Adobe Acrobat 6.0', 'creationdate': '2006-10-18T

In [91]:
# retriever sentence striper funtion
import re
def decompose_to_sentence(text: str) -> list[str]:
    text = re.sub(r"\s+"," ",text).strip()
    sentence = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentence if len(s.strip()) > 20]


In [92]:
# Filter

class KeepOrDrop(BaseModel):
    keep: bool = Field(description="Give Return True if the sentence is relevant, otherwise False")

parser = JsonOutputParser(pydantic_object=KeepOrDrop)

from langchain_core.output_parsers import StrOutputParser

# প্রম্পট পরিবর্তন (JSON এর বদলে YES/NO চাইছি)
filter_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", 
         "You are a strict relevance filter. \n"
         "Does the sentence directly help answer the question? \n"
         "Reply with ONLY one word: YES or NO. Nothing else."
        ),
        ("human", "Question: {question}\n\nSentence:\n{sentence}")
    ]
) 

# নতুন filter_chain (JsonOutputParser এর বদলে StrOutputParser)
filter_chain = filter_prompt | llm | StrOutputParser() | (lambda x: "YES" in x.strip().upper())



In [93]:

# refining (decompose -> filter -> recompose)
def refine(state: State) -> State:
    q = state["question"]
    # Combine retrieve docs into the one contex string
    lenth = len(state['docs'])
    keep_refined_chunks = []
    
    all_kept_strips = []
    for x in range(lenth):
        kept: list[str] = []
        context = state['docs'][x].page_content
        strips = decompose_to_sentence(context)

    #2 FILTER: keep only relevance strips
        for s in strips:
            if filter_chain.invoke({'question':q,"sentence":s}):
                kept.append(s)

        #3 RECOMPOSE: strips merge
        refined_context = "\n".join(kept).strip()

        keep_refined_chunks.append({
            "chunk_no":x,
            "chunk":context,
            "strips":strips,
            "kept_strips": kept,
            "refined_context":refined_context
        })
    
    return {
    "kept_strips": all_kept_strips,
    "refined_context": "\n".join(all_kept_strips).strip()
}

In [94]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        ('system',"Answer only from the context. If not in contex, say you don't know",),
        ('human', "Question : {question}\n\nContext:\n{context}")
    ]
)

def generate(state: State) -> State:
    out = (answer_prompt | llm).invoke({
        "question": state["question"], 
        "context": state['refined_context']  # ✅ ঠিক করা হলো
    })
    return {"answer": out.content}


In [95]:
g = StateGraph(State)
g.add_node("retrieve", retrieve)
g.add_node("refine", refine)
g.add_node("generate", generate)

g.add_edge(START, "retrieve")
g.add_edge("retrieve", "refine")    # ✅ refine যোগ হলো
g.add_edge("refine", "generate")   # ✅ সঠিক ক্রম
g.add_edge("generate", END)

app = g.compile()
g = StateGraph(State)
g.add_node("retrieve", retrieve)
g.add_node("refine", refine)
g.add_node("generate", generate)

g.add_edge(START, "retrieve")
g.add_edge("retrieve", "refine")    # ✅ refine যোগ হলো
g.add_edge("refine", "generate")   # ✅ সঠিক ক্রম
g.add_edge("generate", END)

app = g.compile()


In [98]:
res = app.invoke({"question":"whte is a transformer","docs":[],'answer':""})

print(res["answer"])

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-69c7b80c-777e282e77ae54a10da077c2;6f9f59de-1769-48d3-b1be-cb14299857d2)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

In [ ]:
print(res["docs"][0].page_content)
print('*'*100)
print(res["docs"][1].page_content)
print('*'*100)
print(res["docs"][2].page_content) 
print('*'*100)
print(res["docs"][3].page_content)
print('*'*100)